# Task 3 – Notebook 4: hybrid DNN assignment + KKT water-filling
Keep the DNN's RB assignment, discard its power output and recompute the powers with the same KKT water-filling used in method 4 (water level $L_u=\max(L,\lambda_u)$, total power $=P_{\max}$, $R_{\min}$ guaranteed whenever the assignment admits it).  Inference cost = DNN forward pass + water-filling.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().resolve().parents[1]
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
torch.set_num_threads(1)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from task3_resource.system import P, dbm_to_w, draw_channels, rb_rates, user_rates, POWER_DBM_GRID
from task3_resource import allocation as al, dnn, experiments as ex
T = ex.RES / "tables"; F = ex.RES / "figures"


In [2]:
net = dnn.load(ex.MODELS / "dnn_alloc.pt")
V = draw_channels(5, np.random.default_rng(11)); a = V[0]; pmax = dbm_to_w(25)
x, p_dnn, _ = dnn.predict(net, a, pmax)
p_hyb, info = al.kkt_waterfilling(x, a, pmax)
for n, p_ in [("DNN", p_dnn), ("Hybrid", p_hyb)]:
    R = user_rates(x, p_, a)/1e6
    print(n, "sum %.2f Mbps | users>=1Mbps: %d/8 | power %.3f of budget" % (R.sum(), (R >= 1).sum(), p_.sum()/pmax), "| QoS feasible flag:", info["qos_feasible"] if n == "Hybrid" else "n/a")

DNN sum 30.77 Mbps | users>=1Mbps: 6/8 | power 1.000 of budget | QoS feasible flag: n/a
Hybrid sum 30.45 Mbps | users>=1Mbps: 6/8 | power 1.000 of budget | QoS feasible flag: False


In [3]:
S = pd.read_csv(T / "task3_summary_all_methods.csv")
sel = S[S.method.str.startswith(("4", "5", "6")) & S.power_dbm.isin([15, 20, 25, 30, 35, 40])]
display(sel.pivot(index="power_dbm", columns="method", values="sum_rate_mbps").round(2))
display(sel.pivot(index="power_dbm", columns="method", values="qos_pct").round(1))
display(sel.pivot(index="power_dbm", columns="method", values="all_users_qos_pct").round(1))

method,4 MILP + KKT water-filling,5 DNN,6 Hybrid DNN + KKT water-filling
power_dbm,,,
15.0,13.15,16.36,15.36
20.0,16.51,20.23,19.14
25.0,21.24,24.14,23.19
30.0,27.86,28.03,27.55
35.0,34.62,32.07,31.96
40.0,40.19,36.22,36.21


method,4 MILP + KKT water-filling,5 DNN,6 Hybrid DNN + KKT water-filling
power_dbm,,,
15.0,78.1,49.3,67.2
20.0,90.5,60.0,76.1
25.0,97.9,70.8,84.4
30.0,99.8,79.7,88.8
35.0,100.0,87.3,91.1
40.0,100.0,90.5,91.7


method,4 MILP + KKT water-filling,5 DNN,6 Hybrid DNN + KKT water-filling
power_dbm,,,
15.0,8.0,0.0,0.7
20.0,39.0,0.0,5.7
25.0,83.0,3.7,20.0
30.0,98.7,11.0,35.0
35.0,99.7,29.7,45.7
40.0,99.7,40.7,44.7
